In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np
import pandas as pd 
import os
from typing import Optional, Dict, Any, Tuple, List
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import optuna
import warnings
import polars as pl
import time
import math
import gc
import lightgbm as lgb
from scipy.stats import spearmanr
import joblib
import kaggle_evaluation.default_inference_server
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="lightgbm")
from IPython.display import display, Markdown
from scipy.stats import spearmanr
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hull-tactical-market-prediction/train.csv
/kaggle/input/hull-tactical-market-prediction/test.csv
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_inference_server.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/templates.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/base_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/relay.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/kaggle_evaluation.proto
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2_grpc.py
/kaggl

# Functions

In [2]:
# Lag Features
def add_lag_features(df: pd.DataFrame, cols, lags):
    for col in cols:
        if col not in df.columns:
            continue
        for lag in lags:
            df[f"{col}_lag_{lag}"] = df[col].shift(lag)
    return df

# Rolling Window Features
def add_rolling_features(df: pd.DataFrame, cols, windows):
    for col in cols:
        if col not in df.columns:
            continue
        for w in windows:
            df[f"{col}_roll_mean_{w}"] = df[col].rolling(w, min_periods=1).mean()
            df[f"{col}_roll_std_{w}"] = df[col].rolling(w, min_periods=1).std()
            df[f"{col}_roll_min_{w}"] = df[col].rolling(w, min_periods=1).min()
            df[f"{col}_roll_max_{w}"] = df[col].rolling(w, min_periods=1).max()
            df[f"{col}_roll_median_{w}"] = df[col].rolling(w, min_periods=1).median()
    return df

# Percentage Change Features (Volatility / Momentum)
def add_pct_change_features(df: pd.DataFrame, cols, periods=[1, 3, 7]):
    for col in cols:
        if col not in df.columns:
            continue
        for p in periods:
            df[f"{col}_pct_change_{p}"] = df[col].pct_change(periods=p)
    return df

# First Difference
def add_diff_features(df: pd.DataFrame, cols):
    for col in cols:
        if col in df.columns:
            df[f"{col}_diff_1"] = df[col].diff(1)
            df[f"{col}_diff_7"] = df[col].diff(7)
    return df

# Exponential Weighted Moving Features
def add_ewm_features(df: pd.DataFrame, cols, spans=[7, 14, 30]):
    for col in cols:
        if col not in df.columns:
            continue
        for s in spans:
            df[f"{col}_ewm_mean_{s}"] = df[col].ewm(span=s, adjust=False).mean()
            df[f"{col}_ewm_std_{s}"] = df[col].ewm(span=s, adjust=False).std()
    return df

# Feature Interactions (non-linear relationships)
def add_interaction_features(df: pd.DataFrame, cols):
    numeric_cols = [c for c in cols if df[c].dtype != "object"]
    
    for i, c1 in enumerate(numeric_cols):
        for c2 in numeric_cols[i+1:]:
            df[f"{c1}_x_{c2}"] = df[c1] * df[c2]
            df[f"{c1}_div_{c2}"] = df[c1] / (df[c2] + 1e-7)
    return df

# Rolling Normalization (z-score inside windows)
def add_rolling_normalized_features(df: pd.DataFrame, cols, windows=[14, 30]):
    for col in cols:
        if col not in df.columns:
            continue
        
        for w in windows:
            m = df[col].rolling(w, min_periods=1).mean()
            s = df[col].rolling(w, min_periods=1).std()
            df[f"{col}_roll_zscore_{w}"] = (df[col] - m) / (s + 1e-7)
    return df

# FEATURE PIPELINE
def create_features(df: pd.DataFrame) -> pd.DataFrame:

    """
    Feature engineering pipeline:
    1. Use the columns in TOP_FEATURES_FOR_FE as the base features.
    2. Apply all feature engineering functions in sequence.
    3. Handle missing values: Forward filling → Backward filling → Median filling 
    Note: Define TOP_FEATURES_FOR_FE, LAG_PERIODS, and ROLLING_WINDOWS in advance. 
    """
    
    # Define primary columns for FE
    cols = TOP_FEATURES_FOR_FE
    
    # --- Base FE ---
    df = add_lag_features(df, cols, LAG_PERIODS)
    df = add_rolling_features(df, cols, ROLLING_WINDOWS)

    # --- Extra FE modules ---
    df = add_pct_change_features(df, cols)
    df = add_diff_features(df, cols)
    df = add_ewm_features(df, cols)
    df = add_interaction_features(df, cols)
    df = add_rolling_normalized_features(df, cols)

    df.ffill(inplace=True)
    df.bfill(inplace=True)
    
    for c in df.columns:
        if df[c].isnull().any():
            df[c].fillna(df[c].median(), inplace=True)

    return df

# Config

In [3]:
TRAIN_PATH = '/kaggle/input/hull-tactical-market-prediction/train.csv'

TOP_FEATURES_FOR_FE =['E13', 'I4', 'P13', 'V1', 'P1', 'S12', 'E10', 
                      'M9', 'M12', 'M18', 'I1', 'M17', 'E20', 'E1', 
                      'S10', 'E14', 'V3', 'S6', 'E5', 'M2', 'S11', 
                      'E18', 'I3', 'D7', 'S7', 'E16', 'E12', 'E11', 'M11', 'P7']

LAG_PERIODS = [1, 3, 5, 7, 14, 20]
ROLLING_WINDOWS = [3, 6, 10, 20, 60]

TARGET = 'market_forward_excess_returns'
COLS_TO_DROP = ['forward_returns', 'risk_free_rate', 'E7', 'V10', 'S3', 'M1', 'M14', 'M13', 'M6', 'V9']
TUNER_SEED = 2
N_TRIALS_LIGHTGBM = 10 
N_SPLITS = 4  

# Inference

In [4]:
print("Loading models and features for inference...")

try:
    # Load LightGBM artifacts
    lgb_model = joblib.load("/kaggle/input/kaggle-prepare/final_lgbm.joblib")
    lgb_features = joblib.load("/kaggle/input/kaggle-prepare/features_lgbm.joblib")

except Exception as e:
    raise RuntimeError(
        f"Could not load model/features. Error: {e}"
    )

print("Loading train data...")
history_df = pd.read_csv(TRAIN_PATH)

# Clean history
history_df = history_df.iloc[1006:].reset_index(drop=True)
history_df.drop(columns=[col for col in COLS_TO_DROP if col in history_df.columns and col != TARGET], inplace=True)

if 'date_id' not in history_df.columns:
    history_df['date_id'] = history_df.index

# Screening out the top 25% of the best-performing stocks 
# A conservative stock selection strategy
quantile=0.75
threshold = np.quantile(history_df[TARGET], quantile)
threshold = max(0, threshold)  # Ensure that the threshold is not less than

print("Setup complete. Ready for prediction.")

# Convert return to strategy
def convert_to_strategy(predict: float, threshold: float) -> int:
    """
    Convert predicted return into discrete signal (0,1,2) 
    using a dynamic thresholding based on training distribution.
    """
    if predict <= 0:
        return 0
    elif predict <= threshold:
        return 1
    else:
        return 2

# Main prediction function
def predict(test_df_pl: pl.DataFrame) -> float:
    global history_df

    # Convert to pandas
    test_df_pd = test_df_pl.to_pandas()

    # Assign next date_id
    if 'date_id' not in test_df_pd.columns:
        last_id = history_df['date_id'].max() if not history_df.empty else -1
        test_df_pd['date_id'] = last_id + 1

    # Update historical data
    history_df = pd.concat([history_df, test_df_pd], ignore_index=True)

    # Prepare rolling window slice
    slice_size = max(ROLLING_WINDOWS) + max(LAG_PERIODS) + 5
    window = history_df.tail(slice_size)

    processed = create_features(window)

    # Prepare features for each model
    x_lgb = processed.tail(1)[lgb_features]
    
    # Make predictions
    pred_lgb = float(lgb_model.predict(x_lgb)[0])

    # Convert to discrete signal
    signal = convert_to_strategy(pred_lgb, threshold)

    gc.collect()
    return float(signal)


# -----------------------------
# Inference server
# -----------------------------
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    print("Serving predictions for the competition...")
    inference_server.serve()
else:
    print("Running local gateway for testing...")
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

print("Submission script finished.")

Loading models and features for inference...
Loading train data...
Setup complete. Ready for prediction.
Running local gateway for testing...
Submission script finished.
